In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import faiss
import time

/home/intern/MyWork/rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2003.67it/s]


In [65]:
with open("500DaysofSummer.txt", "r", encoding="utf-8") as file:
    text = file.read()

with open("500DaysofSummer.txt", "r", encoding="utf-8") as f:
    screenplay = f.read()

In [66]:
section_size = 12000  # characters

sections = []

for i in range(0, len(screenplay), section_size):
    sections.append(screenplay[i:i+section_size])

print("Total sections:", len(sections))

Total sections: 9


In [6]:
def chunk_text(text, chunk_size=800, overlap=300):
    chunks = []

    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks
chunks = chunk_text(text)

In [8]:
emb = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

In [69]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY2")

In [70]:
from groq import Groq

client = Groq(api_key=api_key)

In [53]:
query = "What was douchebag referring to in the movie?"
query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = query_embedding.reshape(1, -1).astype("float32")
retrieved_chunks = []
distances, indices = index.search(query_embedding, 10)
for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

distances, indices = index.search(query_embedding, 10)

In [54]:
prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{query}

"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("Question:" ,query)
print("Answer:" ,response.choices[0].message.content)

Question: What was douchebag referring to in the movie?
Answer: The Douchebag was referring to Tom when he said "You’re serious? This guy?" implying that he was surprised Summer was with Tom. 
The supporting sentence is: 
DOUCHE
You’re serious? This guy?


In [33]:
questions = [
    "Who is Tom?",
    "Who is Summer?",
    "Where does Tom work?",
    "What city does the story take place in?",
    "Who is Rachel?",
    "Why is Tom sad?",
    "What band do Tom and Summer like?",
    "What happens at IKEA?",
    "Who is Millie?",
    "How does the movie end?"
]

In [41]:
for q in questions:
    print("="*80)
    print("Question:", q)
    # Retrieval
    query_embedding = model.encode([q]).astype("float32")
    distances, indices = index.search(query_embedding, k=12)

    # print("\nRetrieved Chunks:")
    # for rank, idx in enumerate(indices[0]):
    #     print(f"\n--- Chunk {idx} ---")
    #     print(chunks[idx][:500])

    context = "\n\n".join(chunks[idx] for idx in indices[0])

    prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{q}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    print("\nAnswer:")
    print(response.choices[0].message.content)
    time.sleep(1)

Question: Who is Tom?

Answer:
Tom.

Supporting sentence: The first sentence of the provided context states: "t isn’t big, it is organized by a master. Two, on the walls, is a series of framed portraits, each one a famous building and its architectural blueprint." The context later refers to "Tom" in various places, clearly showing that "t" stands for Tom.

However this is confirmed in the following sentence where 'TOM' is used as a name: 
GIRL - Thomas.
TOM freezes.
Question: Who is Summer?

Answer:
Summer is a woman.

(Throughout the following, SUBTITLES will reveal specifics of the Narrator’s points.)
NARRATOR
Summer Finn was a woman.
 
FREEZE on SUMMER. (Throughout the following, SUBTITLES will
reveal specifics of the Narrator’s points.)
NARRATOR
Height: average.
Titles reveal specifics: 5’ 5”
NARRATOR
Weight: average.
Titles: 121 pounds.
NARRATOR
Shoe size: slightly above average.
Titles: Size 8.
NARRATOR
For all intents and purposes,
Summer Finn... just another girl.
Question: Wh

In [45]:
for i, chunk in enumerate(chunks):

    prompt = f"""
Generate 5 Question-Answer pairs.

Rules:
- Answer must be directly supported by the text.
- Keep answers concise.
- Do not invent anything.

More Rules:
1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."

Return JSON like this:

[
  {{
    "question": "",
    "answer": ""
  }}
]

Text:

{chunk}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    print(f"\n===== Chunk {i} =====")
    print(response.choices[0].message.content)


===== Chunk 0 =====
[
  {
    "question": "Who are the authors of the screenplay?",
    "answer": "Scott Neustadter and Michael H. Weber"
  },
  {
    "question": "What is the name of the narrator?",
    "answer": "No information available in the provided context"
  },
  {
    "question": "Where are the story's key events taking place?",
    "answer": "Downtown Los Angeles, CA"
  },
  {
    "question": "Who are the main characters in the story?",
    "answer": "Tom and Summer"
  },
  {
    "question": "What is the nature of Tom's relationship with Summer?",
    "answer": "Tom has met Summer"
  }
]

===== Chunk 1 =====
[
  {
    "question": "What year was Tom a pre-teen?",
    "answer": "1989"
  },
  {
    "question": "Where did Tom Hansen grow up?",
    "answer": "Margate, New Jersey"
  },
  {
    "question": "Why did Tom grow up believing he 'd never truly be happy?",
    "answer": "Until the day he met 'the one'."
  },
  {
    "question": "Where is Tom sitting at?",
    "answer": "I

In [55]:
import re
import json

with open("testData.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Find every JSON array
arrays = re.findall(r"\[\s*{.*?}\s*\]", text, flags=re.DOTALL)

dataset = []

for arr in arrays:
    try:
        dataset.extend(json.loads(arr))
    except Exception:
        print("Skipped one malformed block.")

print("Questions:", len(dataset))

with open("testData.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

print("Done!")

Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Skipped one malformed block.
Questions: 924
Done!


In [63]:
prompt = f"""
You are an expert dataset creator for evaluating Retrieval-Augmented Generation (RAG) systems.

Your task is to generate EXACTLY 100 high-quality Question-Answer pairs from the screenplay below.

========================
RULES
========================

1. Every answer MUST be directly supported by the screenplay.

2. NEVER invent information.

3. NEVER use outside knowledge.

4. Cover the ENTIRE screenplay.

5. Questions should evaluate whether a RAG system retrieved the correct context.

6. Every question should have ONE clear answer.

7. Answers should be short (1-2 sentences maximum).

8. Do NOT repeat questions.

9. Do NOT ask nearly identical questions.

10. If information is not explicitly written, DO NOT create a question about it.

========================
QUESTION TYPES
========================

Generate a good mixture of:

• Characters
• Relationships
• Character motivations
• Character emotions
• Occupations
• Places
• Timeline
• Events
• Conversations
• Songs
• Movies referenced
• Objects
• Important actions
• Cause and effect
• Story progression
• Character development

========================
DIFFICULTY
========================

20 Easy
50 Medium
30 Hard

Easy:
Single fact retrieval.

Medium:
Requires combining 2-3 nearby facts.

Hard:
Requires understanding events across multiple scenes,
but the answer MUST still be explicitly supported by the screenplay.

========================
DO NOT ASK ABOUT
========================

❌ Page numbers

❌ Revision dates

❌ Screenplay formatting

❌ Scene numbers

❌ INT / EXT

❌ Typography

❌ File metadata

❌ Chapter headings

❌ Copyright notices

❌ Revision colors

❌ Anything outside the story

========================
GOOD EXAMPLES
========================

✓ Why does Tom believe in true love?

✓ What company does Tom work for?

✓ What advice does Rachel give Tom?

✓ What song is Summer singing?

✓ Why does Summer cut her hair?

✓ What happens during the IKEA scene?

✓ Why is Tom heartbroken?

✓ What career does Tom actually want?

========================
BAD EXAMPLES
========================

✗ What page does this happen on?

✗ What revision date is written?

✗ What scene number is this?

✗ What is written at the top of the script?

✗ What year was the screenplay revised?

========================
OUTPUT FORMAT
========================

Return ONLY valid JSON.

No markdown.

No explanations.

No extra text.

Use this exact format:

[
    {{
        "question": "...",
        "answer": "..."
    }}
]

========================
SCREENPLAY
========================

{text}
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kxyz7zxhetmsj193cm6snge2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 30952, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [61]:
print(text)

(500) DAYS OF SUMMER
by
Scott Neustadter
&
Michael H. Weber
April 16, 2008NOTE: THE FOLLOWING IS A WORK OF FICTION. ANY RESEMBLANCE TO
PERSONS LIVING OR DEAD IS PURELY COINCIDENTAL.ESPECIALLY YOU JENNY BECKMAN.BITCH.FADE IN:
A single number in parenthesis, exactly like so:
(488)
EXT. ANGELUS PLAZA — DOWNTOWN LOS ANGELES, CA — DAY 1
And we’re looking at a MAN (20s) and a WOMAN (20s) ona
bench, high above the city of Los Angeles. Their names are
TOM and SUMMER and right now neither one says a word.
CLOSE ON their HANDS, intertwined. Notice the wedding ring
on her finger. CLOSE ON Tom, looking at Summer the way every
woman wants to be looked at.
And then a DISTINGUISHED VOICE begins to speak to us.
NARRATOR
This is a story of boy meets girl.
INT CONFERENCE ROOM —- DAY 2
TOM HANSEN sits at a very long rectangular conference table.
The walls are lined with framed blow-up sized greeting cards.
Tom, dark hair and blue eyes, wears a t-shirt under his
sports coat and Adidas tennis shoes to bala

In [72]:
import json

dataset = []

for i, section in enumerate(sections):

    print(f"\nGenerating questions for section {i+1}/{len(sections)}")

    prompt = f"""
You are an expert dataset creator for evaluating Retrieval-Augmented Generation (RAG) systems.

Your task is to Generate EXACTLY 20 Question-Answer pairs from the screenplay below.

========================
RULES
========================

1. Every answer MUST be directly supported by the screenplay.

2. NEVER invent information.

3. NEVER use outside knowledge.

4. Cover as much of THIS screenplay section as possible.

5. Questions should evaluate whether a RAG system retrieved the correct context.

6. Every question should have ONE clear answer.

7. Answers should be short (1-2 sentences maximum).

8. Do NOT repeat questions.

9. Do NOT ask nearly identical questions.

10. If information is not explicitly written, DO NOT create a question about it.

11. Prefer meaningful story questions over trivial object or location questions.

12. Avoid asking multiple questions about the same event unless they test different information.

13. Do not create questions whose answer is simply "yes" or "no".

14. Include the minimum information necessary in the answer, but make it complete.


========================
QUESTION TYPES
========================

Generate a good mixture of:

• Characters
• Relationships
• Character motivations
• Character emotions
• Occupations
• Places
• Timeline
• Events
• Conversations
• Songs
• Movies referenced
• Objects
• Important actions
• Cause and effect
• Story progression
• Character development

========================
DIFFICULTY
========================

Generate a good mixture of difficulty levels:
• 5 Easy
• 10 Medium
• 5 Hard

Easy:
Single fact retrieval.

Medium:
Requires combining 2-3 nearby facts.

Hard:
Requires understanding multiple events within THIS screenplay section,
but the answer must still be explicitly supported by the text.

========================
DO NOT ASK ABOUT
========================

❌ Page numbers

❌ Revision dates

❌ Screenplay formatting

❌ Scene numbers

❌ INT / EXT

❌ Typography

❌ File metadata

❌ Chapter headings

❌ Copyright notices

❌ Revision colors

❌ Anything outside the story

========================
GOOD EXAMPLES
========================

✓ Why does Tom believe in true love?

✓ What company does Tom work for?

✓ What advice does Rachel give Tom?

✓ What song is Summer singing?

✓ Why does Summer cut her hair?

✓ What happens during the IKEA scene?

✓ Why is Tom heartbroken?

✓ What career does Tom actually want?

========================
BAD EXAMPLES
========================

✗ What page does this happen on?

✗ What revision date is written?

✗ What scene number is this?

✗ What is written at the top of the script?

✗ What year was the screenplay revised?

========================
OUTPUT FORMAT
========================

Return ONLY valid JSON.

No markdown.

No explanations.

No extra text.

Use this exact format:

[
    {{
        "id": 1,
        "question": "...",
        "answer": "..."
    }}
]

========================
SCREENPLAY
========================

{section}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",max_completion_tokens=3500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    try:
        qa_pairs = json.loads(response.choices[0].message.content)
        dataset.extend(qa_pairs)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON for section {i+1}")
    time.sleep(20)

    with open("test_dataset.json", "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=4, ensure_ascii=False)

print("Saved", len(dataset), "questions.")


Generating questions for section 1/9

Generating questions for section 2/9

Generating questions for section 3/9

Generating questions for section 4/9

Generating questions for section 5/9

Generating questions for section 6/9

Generating questions for section 7/9

Generating questions for section 8/9

Generating questions for section 9/9
Saved 180 questions.


In [83]:
import json

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric
)

In [85]:
with open("test_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [87]:
faithfulness = FaithfulnessMetric(model="llama-3.3-70b-versatile")
answer_relevancy = AnswerRelevancyMetric()
hallucination = HallucinationMetric()

DeepEvalError: OpenAI API key is not configured. Set OPENAI_API_KEY in your environment or pass `api_key` to GPTModel(...).